# Mobility Network & POI Analysis

This notebook focuses on the **structure** of your mobility:
*   **Network Analysis**: Which roads do you use most?
*   **Place Analysis**: Where do you spend your time?
*   **Route Animation**: visualizing specific movement paths.

All outputs are saved to the `outputs/` folder.


In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from datetime import timedelta
from dateutil import parser as dtparser
import geopandas as gpd
from shapely.geometry import Point, LineString
import osmnx as ox
import networkx as nx
from matplotlib.lines import Line2D

# Ensure output directory exists
os.makedirs('outputs', exist_ok=True)


## 1. Configuration
Set your city and analysis parameters here.


In [ ]:
CITY_NAME = 'Beirut, Lebanon'
BUFFER_KM = 8
NETWORK_TYPE = 'all'  # 'drive', 'walk', 'all'

# Cleaning parameters
MIN_VISIT_MIN = 3      # Ignore stops < 3 mins
MERGE_GAP_MIN = 10     # Merge visits if gap <= 10 mins
DT_CLIP_MAX_MIN = 30   # Max time weight for movement edges (prevents long gaps dominating)
MAX_SPEED_MPS = 60     # Speed filter for activities

# Aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 2. Load & Clean Data
Reading the Google Takeout JSON file.


In [ ]:
path = 'location-history.json'
print(f'Loading {path}...')
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Loaded {len(data)} records.')


In [ ]:
def parse_geo(s: str):
    if not s or s == "geo:0,0": return None
    s = s.replace("geo:", "")
    try:
        lat, lon = s.split(",")
        return float(lat), float(lon)
    except:
        return None

def duration_minutes(t0, t1):
    return (t1 - t0).total_seconds() / 60.0

def normalize_mode(m):
    if not m: return 'UNKNOWN'
    return m.upper().replace('IN_', '').replace('VEHICLE', 'CAR')

rows_visit, rows_act, rows_path = [], [], []

for i, ep in enumerate(data):
    if 'startTime' not in ep or 'endTime' not in ep:
        continue

    start = dtparser.isoparse(ep['startTime'])
    end   = dtparser.isoparse(ep['endTime'])
    dur = duration_minutes(start, end)
    if dur <= 0: continue

    # 1. Visits
    if 'visit' in ep:
        v = ep['visit']
        tc = v.get('topCandidate', {})
        loc = tc.get('placeLocation')
        if loc:
            geo = parse_geo(loc)
            if geo:
                lat, lon = geo
                rows_visit.append({
                    'episode_id': i,
                    'start_time': start,
                    'end_time': end,
                    'duration_min': dur,
                    'lat': lat, 'lon': lon,
                    'place_id': tc.get('placeID'),
                    'semantic_type': tc.get('semanticType'),
                    'visit_prob': float(v.get('probability', 1.0))
                })

    # 2. Timeline Path (Raw Points)
    if 'timelinePath' in ep:
        for p in ep['timelinePath']:
            geo = parse_geo(p['point'])
            if geo:
                plat, plon = geo
                offset = float(p.get('durationMinutesOffsetFromStartTime', 0.0))
                rows_path.append({
                    'episode_id': i,
                    'time': start + timedelta(minutes=offset),
                    'lat': plat, 'lon': plon
                })

visits = pd.DataFrame(rows_visit)
pathpts = pd.DataFrame(rows_path)
print(f'Extracted {len(visits)} visits and {len(pathpts)} path points.')


### Data Cleaning
*   Filter invalid durations.
*   Merge consecutive visits to the same place (removes noise).
*   Convert Timestamps.


In [ ]:
# Timestamp conversion
if not visits.empty:
    visits['start_time'] = pd.to_datetime(visits['start_time'], utc=True)
    visits['end_time'] = pd.to_datetime(visits['end_time'], utc=True)
    visits = visits[visits['duration_min'] >= MIN_VISIT_MIN].copy()
    visits['weight'] = visits['duration_min'] * visits['visit_prob']

    # Merge consecutive visits
    visits = visits.sort_values(['start_time'])
    visits['prev_place'] = visits['place_id'].shift(1)
    visits['prev_end'] = visits['end_time'].shift(1)
    gap = (visits['start_time'] - visits['prev_end']).dt.total_seconds()/60
    merge_mask = (visits['place_id'].notna()) & (visits['place_id'] == visits['prev_place']) & (gap <= MERGE_GAP_MIN)
    visits['group'] = (~merge_mask).cumsum()

    visits = (visits.groupby('group', as_index=False).agg({
        'start_time': 'min', 'end_time': 'max', 'duration_min': 'sum',
        'lat': 'first', 'lon': 'first', 'place_id': 'first', 'semantic_type': 'first',
        'weight': 'sum'
    }))

if not pathpts.empty:
    pathpts['time'] = pd.to_datetime(pathpts['time'], utc=True)

print(f'Visits after merging: {len(visits)}')

# Save summary CSV
visits[['start_time', 'duration_min', 'semantic_type']].to_csv('outputs/visits_summary.csv', index=False)
print('Saved outputs/visits_summary.csv')


## 3. Network Acquisition
Downloading the street network from OpenStreetMap via OSMnx.


In [ ]:
print(f'Downloading network for {CITY_NAME}...')
try:
    # 1. Get base polygon
    city_gdf = ox.geocode_to_gdf(CITY_NAME)
    city_poly = city_gdf.geometry.iloc[0]
    
    # 2. Buffer in metric CRS
    utm_crs = city_gdf.estimate_utm_crs()
    city_poly_m = city_gdf.to_crs(utm_crs).geometry.iloc[0].buffer(BUFFER_KM * 1000)
    city_poly = gpd.GeoSeries([city_poly_m], crs=utm_crs).to_crs('EPSG:4326').iloc[0]
    
    # 3. Download Graph
    G = ox.graph_from_polygon(city_poly, network_type=NETWORK_TYPE, simplify=True)
    G = ox.project_graph(G) # To UTM
    edges = ox.graph_to_gdfs(G, nodes=False, fill_edge_geometry=True).reset_index()
    
    print(f'Network loaded: {len(edges)} edges.')
except Exception as e:
    print(f'Error loading network: {e}')
    raise


## 4. Map Matching
Snap GPS points to the nearest road segments to estimate usage.


In [ ]:
# Project points to same CRS as Graph
def df_to_gdf(df, lat_col='lat', lon_col='lon', crs='EPSG:4326'):
    return gpd.GeoDataFrame(df, geometry=[Point(xy) for xy in zip(df[lon_col], df[lat_col])], crs=crs)

gpath = df_to_gdf(pathpts).to_crs(G.graph['crs'])
# Filter to graph bounds
minx, miny, maxx, maxy = edges.total_bounds
gpath = gpath.cx[minx:maxx, miny:maxy]

print(f'Snapping {len(gpath)} points to nearest edges...')
uvk = ox.distance.nearest_edges(G, X=gpath.geometry.x.values, Y=gpath.geometry.y.values)

# Unpack (u, v, key)
gpath['u'] = [x[0] for x in uvk]
gpath['v'] = [x[1] for x in uvk]
gpath['key'] = [x[2] for x in uvk]

# Calculate time usage per edge
edge_counts = gpath.groupby(['u', 'v', 'key']).size().rename('point_count').reset_index()
edges_w = edges.merge(edge_counts, on=['u', 'v', 'key'], how='left')
edges_w['point_count'] = edges_w['point_count'].fillna(0)
edges_w['log_count'] = np.log1p(edges_w['point_count'])

print('Map matching complete.')


## 5. Visualizations: Road Network
Maps showing which streets you use the most.


In [ ]:
def plot_network_heatmap(edges_gdf, col, title, filename, cmap='inferno'):
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_axis_off()
    ax.set_title(title, fontsize=16)
    
    # Background roads (low alpha)
    edges_gdf.plot(ax=ax, linewidth=0.5, color='#444444', alpha=0.1)
    
    # Activity roads
    active = edges_gdf[edges_gdf[col] > 0]
    if not active.empty:
        # Scale linewidth by activity
        lw = 0.5 + 2 * (active[col] / active[col].max())
        active.plot(ax=ax, column=col, cmap=cmap, linewidth=lw, alpha=0.8)
    
    plt.savefig(f'outputs/{filename}', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved outputs/{filename}')

plot_network_heatmap(edges_w, 'log_count', 'Road Usage Heatmap', 'frequented_roads.png')


## 6. Place & POI Analysis
Identifying specific locations and categories of interest.


In [ ]:
gvisits = df_to_gdf(visits).to_crs(G.graph['crs'])
gvisits = gvisits.cx[minx:maxx, miny:maxy]

# Scale marker size by log duration
gvisits['log_dur'] = np.log1p(gvisits['duration_min'])
sizes = 5 + 20 * (gvisits['log_dur'] / gvisits['log_dur'].max())

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()
edges.plot(ax=ax, linewidth=0.5, alpha=0.15, color='gray')
gvisits.plot(ax=ax, markersize=sizes, column='log_dur', cmap='plasma', alpha=0.7)

plt.title('Places Frequented (Size = Duration)', fontsize=16)
plt.savefig('outputs/places_frequented.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved outputs/places_frequented.png')


### POI Category Analytics
Charts showing the breakdown of visit semantic types.


In [ ]:
if 'semantic_type' in visits.columns:
    type_counts = visits['semantic_type'].value_counts().head(10)
    if not type_counts.empty:
        plt.figure(figsize=(10, 6))
        type_counts.plot(kind='bar', color='teal')
        plt.title('Top 10 Visit Categories')
        plt.ylabel('Number of Visits')
        plt.xticks(rotation=45)
        plt.savefig('outputs/visit_categories_bar.png', bbox_inches='tight')
        plt.close()
        print('Saved outputs/visit_categories_bar.png')
